# CG4002: Voice PyTorch 1D-CNN Trainer & HLS Export

This notebook mirrors the gesture training workflow, but for voice classification.


## 1. Import dependencies & configuration

In [1]:
# Install required packages (uncomment if needed)
%pip install numpy pandas scikit-learn matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Install PyTorch + Torchaudio + Torchcodec (uncomment if needed)
%pip install torch torchaudio torchcodec soundfile


^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import random
import re

import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

# CONFIGURATION
SAMPLE_RATE = 16000
N_MFCC = 40
TARGET_FRAMES = 50
NUM_CLASSES = 3
BATCH_SIZE = 16
EPOCHS = 40
LEARNING_RATE = 0.001
AUGMENT_FACTOR = 3
RANDOM_SEED = 42

AUDIO_ROOT = Path('../data/audio')

def get_latest_audio_folder(root: Path) -> Path:
    candidates = []
    for d in root.iterdir():
        if not d.is_dir():
            continue
        if re.fullmatch(r'\d{8}', d.name):
            try:
                dt = datetime.strptime(d.name, '%d%m%Y')
                candidates.append((dt, d))
            except ValueError:
                pass

    if not candidates:
        raise RuntimeError(f'No dated audio folders found under: {root.resolve()}')

    candidates.sort(key=lambda x: x[0])
    return candidates[-1][1]

AUDIO_DIR = get_latest_audio_folder(AUDIO_ROOT)
print(f'Using latest audio folder: {AUDIO_DIR}')

MANIFEST_CSV = AUDIO_DIR / 'voice_manifest.csv'
FEATURES_NPY = AUDIO_DIR / 'voice_features.npy'
LABELS_NPY = AUDIO_DIR / 'voice_labels.npy'
TRAIN_NPY = AUDIO_DIR / 'voice_X_train.npy'
TEST_NPY = AUDIO_DIR / 'voice_X_test.npy'
YTRAIN_NPY = AUDIO_DIR / 'voice_y_train.npy'
YTEST_NPY = AUDIO_DIR / 'voice_y_test.npy'
MEAN_NPY = AUDIO_DIR / 'voice_mean.npy'
STD_NPY = AUDIO_DIR / 'voice_std.npy'
WEIGHTS_H_PATH = AUDIO_DIR / 'voice_cnn_weights.h'

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# CUDA / backend diagnostics
print(f'PyTorch version: {torch.__version__}')
print(f'TorchAudio version: {torchaudio.__version__}')
print(f'PyTorch CUDA build: {torch.version.cuda}')
if hasattr(torchaudio, 'list_audio_backends'):
    print(f'Available torchaudio backends: {torchaudio.list_audio_backends()}')

cuda_available = torch.cuda.is_available()
cuda_count = torch.cuda.device_count() if cuda_available else 0
if cuda_available and cuda_count > 0:
    device = torch.device('cuda')
    print(f'Using device: {device} ({torch.cuda.get_device_name(0)})')
else:
    device = torch.device('cpu')
    if torch.version.cuda is None:
        print('Using device: cpu (reason: installed PyTorch build has no CUDA support)')
    else:
        print('Using device: cpu (reason: CUDA not available to this kernel/driver)')


## 2. Data processing & augmentation

### 2.1 Scan and label raw voice files

In [4]:
# Expected folder layout:
# ../data/audio/
#   class0/
#     *.wav
#   class1/
#     *.wav
#   class2/
#     *.wav

def scan_audio_dataset(root: Path):
    if not root.exists():
        raise FileNotFoundError(f'Audio folder not found: {root.resolve()}')

    class_dirs = sorted([p for p in root.iterdir() if p.is_dir()])
    if len(class_dirs) == 0:
        raise RuntimeError(f'No class subfolders found in: {root.resolve()}')

    class_map = {d.name: i for i, d in enumerate(class_dirs)}

    rows = []
    for class_dir in class_dirs:
        label_id = class_map[class_dir.name]
        for wav in sorted(class_dir.glob('*.wav')):
            rows.append({
                'path': str(wav),
                'label': class_dir.name,
                'label_id': label_id,
            })

    if len(rows) == 0:
        raise RuntimeError('No .wav files found under class folders.')

    manifest = pd.DataFrame(rows)
    return manifest, class_map

manifest_df, CLASS_MAP = scan_audio_dataset(AUDIO_DIR)
manifest_df.to_csv(MANIFEST_CSV, index=False)

print(f'✅ Found {len(manifest_df)} voice samples')
print('Class map:', CLASS_MAP)
print(manifest_df.head())


✅ Found 3000 voice samples
Class map: {'go': 0, 'no': 1, 'yes': 2}
                                              path label  label_id
0  ..\data\audio\18022026\go\004ae714_nohash_0.wav    go         0
1  ..\data\audio\18022026\go\0132a06d_nohash_2.wav    go         0
2  ..\data\audio\18022026\go\0137b3f4_nohash_0.wav    go         0
3  ..\data\audio\18022026\go\0137b3f4_nohash_2.wav    go         0
4  ..\data\audio\18022026\go\0137b3f4_nohash_3.wav    go         0


### 2.2 Convert waveform to fixed MFCC windows

In [5]:
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=N_MFCC,
    melkwargs={
        'n_fft': 512,
        'win_length': 400,
        'hop_length': 160,
        'n_mels': 40,
        'center': True,
        'power': 2.0,
    }
)

def fix_mfcc_time(mfcc_2d, target_frames=TARGET_FRAMES):
    # mfcc_2d: [40, T]
    t = mfcc_2d.shape[1]
    if t == target_frames:
        return mfcc_2d
    if t > target_frames:
        return mfcc_2d[:, :target_frames]
    pad = target_frames - t
    return torch.nn.functional.pad(mfcc_2d, (0, pad), mode='constant', value=0.0)


def load_wav_to_mfcc(path):
    wav, sr = torchaudio.load(path)  # [ch, n]
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)

    feat = mfcc_transform(wav).squeeze(0)  # [40, T]
    feat = fix_mfcc_time(feat, TARGET_FRAMES)

    # Per-coefficient normalization across time
    mu = feat.mean(dim=1, keepdim=True)
    sd = feat.std(dim=1, keepdim=True).clamp_min(1e-6)
    feat = (feat - mu) / sd

    return feat.numpy().astype(np.float32)


### 2.3 Audio augmentation

In [16]:
def augment_waveform(wav, noise_std=0.003, gain_low=0.85, gain_high=1.15):
    # wav shape: [1, N]
    aug = wav.clone()

    # Random gain
    gain = torch.empty(1).uniform_(gain_low, gain_high).item()
    aug = aug * gain

    # Additive gaussian noise
    noise = torch.randn_like(aug) * noise_std
    aug = aug + noise

    return aug.clamp(-1.0, 1.0)

def load_wav_safe(path):
    p = Path(path).resolve()

    # Try explicit soundfile backend first (most reliable for wav on Windows).
    try:
        return torchaudio.load(str(p), backend='soundfile')
    except Exception:
        pass

    # Fall back to torchaudio default backend selection.
    try:
        return torchaudio.load(str(p))
    except Exception:
        pass

    # Final fallback: decode with pysoundfile and convert to torch tensor.
    import soundfile as sf
    audio, sr = sf.read(str(p), dtype='float32')
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim == 1:
        wav = torch.from_numpy(audio).unsqueeze(0)
    else:
        wav = torch.from_numpy(audio.T)
    return wav, int(sr)

all_X = []
all_y = []

for _, row in manifest_df.iterrows():
    wav_path = row['path']
    label_id = int(row['label_id'])

    wav, sr = load_wav_safe(wav_path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)

    # original + augmented variants
    versions = [wav]
    for _ in range(AUGMENT_FACTOR - 1):
        versions.append(augment_waveform(wav))

    for v in versions:
        feat = mfcc_transform(v).squeeze(0)
        feat = fix_mfcc_time(feat, TARGET_FRAMES)
        mu = feat.mean(dim=1, keepdim=True)
        sd = feat.std(dim=1, keepdim=True).clamp_min(1e-6)
        feat = (feat - mu) / sd

        all_X.append(feat.numpy().astype(np.float32))   # [40, 50]
        all_y.append(label_id)

X = np.stack(all_X, axis=0)  # [N, 40, 50]
y = np.array(all_y, dtype=np.int64)

np.save(FEATURES_NPY, X)
np.save(LABELS_NPY, y)

print('Features generated')
print('X shape:', X.shape)
print('y shape:', y.shape)
print('Class counts:', {int(i): int((y == i).sum()) for i in np.unique(y)})


RuntimeError: Couldn't find appropriate backend to handle uri ..\data\audio\18022026\go\004ae714_nohash_0.wav and format None.

## 3. Model definition & training

### 3.1 Load Data

In [12]:
X = np.load(FEATURES_NPY).astype(np.float32)
y = np.load(LABELS_NPY).astype(np.int64)

# Split with stratification
idx = np.arange(len(y))
train_idx, test_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y,
)

X_train = X[train_idx]
X_test = X[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

# Dataset-level normalization (fit on train only)
train_mean = X_train.mean(axis=(0, 2), keepdims=True)  # [1,40,1]
train_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6

X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

# Save normalization (flatten to 40 for deployment convenience)
np.save(MEAN_NPY, train_mean.reshape(-1).astype(np.float32))
np.save(STD_NPY, train_std.reshape(-1).astype(np.float32))

np.save(TRAIN_NPY, X_train)
np.save(TEST_NPY, X_test)
np.save(YTRAIN_NPY, y_train)
np.save(YTEST_NPY, y_test)

print(f'Raw shapes: X_train={X_train.shape}, X_test={X_test.shape}')
print(f'Saved train/test split and mean/std to: {AUDIO_DIR}')

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False)


Raw shapes: X_train=(7200, 40, 50), X_test=(1800, 40, 50)
Saved train/test split and mean/std to: ..\data\audio\18022026


### 3.2 Model Architecture

In [10]:
class VoiceCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(VoiceCNN, self).__init__()

        # Input: [B, 40, 50]
        self.conv1 = nn.Conv1d(in_channels=40, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)  # 50 -> 25

        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.AdaptiveAvgPool1d(1)      # 25 -> 1

        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.squeeze(-1)  # [B, 32]
        x = self.fc(x)     # [B, C]
        return x

class VoiceCNN_Small(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, c1=12, c2=16):
        super().__init__()
        # Input: [B, 40, 50]
        self.conv1 = nn.Conv1d(40, c1, kernel_size=3, padding=1, bias=True)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool1d(kernel_size=2)   # 50 -> 25

        self.conv2 = nn.Conv1d(c1, c2, kernel_size=3, padding=1, bias=True)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.AdaptiveAvgPool1d(1)       # 25 -> 1

        self.fc = nn.Linear(c2, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))  # [B, c1, 25]
        x = self.pool2(self.relu2(self.conv2(x)))  # [B, c2, 1]
        x = x.squeeze(-1)                          # [B, c2]
        x = self.fc(x)                             # [B, C]
        return x

model = VoiceCNN_Small(NUM_CLASSES).to(device)
print(f'✅ Model Initialized on {device}')
print(model)


✅ Model Initialized on cuda
VoiceCNN_Small(
  (conv1): Conv1d(40, 12, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu1): ReLU(inplace=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(12, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu2): ReLU(inplace=True)
  (pool2): AdaptiveAvgPool1d(output_size=1)
  (fc): Linear(in_features=16, out_features=3, bias=True)
)


### 3.3 Training Loop

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f'🚀 Starting training for {EPOCHS} epochs...')

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / max(total, 1)
    avg_loss = running_loss / max(len(train_loader), 1)

    model.eval()
    test_correct = 0
    test_total = 0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
            y_true.extend(labels.detach().cpu().numpy().tolist())
            y_pred.extend(predicted.detach().cpu().numpy().tolist())

    test_acc = 100 * test_correct / max(test_total, 1)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.1f}% | Test Acc: {test_acc:.1f}%')

print('✅ Training Complete.')

final_acc = accuracy_score(y_true, y_pred) * 100.0
print(f'Final Test Accuracy: {final_acc:.2f}%')
print(classification_report(y_true, y_pred, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Voice CNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()


🚀 Starting training for 40 epochs...


NameError: name 'train_loader' is not defined

## 4. Export to C++ Header (`voice_cnn_weights.h`)
This extracts trained parameters from `model.state_dict()` and formats them for HLS code.

In [ ]:
def export_pytorch_weights(model, filename=WEIGHTS_H_PATH, fuse_input_norm=False, norm_mean=None, norm_std=None):
    print(f'Exporting weights to {filename}...')

    params = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if fuse_input_norm:
        if norm_mean is None or norm_std is None:
            raise ValueError('norm_mean and norm_std are required when fuse_input_norm=True')

        conv_w_key = 'conv1.weight'
        conv_b_key = 'conv1.bias'
        if conv_w_key not in params or conv_b_key not in params:
            raise KeyError('conv1 weights/bias not found in model state_dict')

        mean_t = torch.as_tensor(norm_mean, dtype=params[conv_w_key].dtype)
        std_t = torch.as_tensor(norm_std, dtype=params[conv_w_key].dtype)

        in_ch = params[conv_w_key].shape[1]
        if mean_t.numel() != in_ch or std_t.numel() != in_ch:
            raise ValueError(f'Expected mean/std with {in_ch} values, got {mean_t.numel()} and {std_t.numel()}')
        if torch.any(std_t == 0):
            raise ValueError('std contains zero; cannot fuse normalization')

        inv_std = (1.0 / std_t).view(1, -1, 1)
        params[conv_w_key] = params[conv_w_key] * inv_std

        # bias' = bias - sum_{c,k}(W/std * mean)
        bias_shift = (params[conv_w_key] * mean_t.view(1, -1, 1)).sum(dim=(1, 2))
        params[conv_b_key] = params[conv_b_key] - bias_shift

        print('Fused z-score normalization into conv1 for raw-input inference.')

    filename = Path(filename)
    with open(filename, 'w') as f:
        f.write('#ifndef VOICE_CNN_WEIGHTS_H\n#define VOICE_CNN_WEIGHTS_H\n\n')
        f.write('#include "voice_typedefs.h"\n\n')

        total_params = 0
        for name, tensor in params.items():
            clean_name = name.replace('.', '_').replace('weight', 'w').replace('bias', 'b')
            data = tensor.numpy().flatten()
            size = len(data)
            total_params += size

            print(f'Processing {name} -> {clean_name} ({size} elements)')

            f.write(f'// PyTorch Layer: {name} (Shape: {tuple(tensor.shape)})\n')
            f.write(f'static const data_t {clean_name}[{size}] = {{\n')

            for i, val in enumerate(data):
                f.write(f'{val:.6f}')
                if i < size - 1:
                    f.write(', ')
                if (i + 1) % 10 == 0:
                    f.write('\n    ')
            f.write('\n};\n\n')

        f.write('#endif // VOICE_CNN_WEIGHTS_H\n')

    print(f'\nDone! Total parameters: {total_params}')
    print(f"File saved as '{filename}'. Upload this to Vitis HLS.")


export_pytorch_weights(
    model,
    WEIGHTS_H_PATH,
    fuse_input_norm=True,
    norm_mean=np.load(MEAN_NPY),
    norm_std=np.load(STD_NPY),
)


Exporting weights to ../data/audio/18022026/voice_cnn_weights.h...
Fused z-score normalization into conv1 for raw-input inference.
Processing conv1.weight -> conv1_w (1920 elements)
Processing conv1.bias -> conv1_b (16 elements)
Processing conv2.weight -> conv2_w (1536 elements)
Processing conv2.bias -> conv2_b (32 elements)
Processing fc.weight -> fc_w (96 elements)
Processing fc.bias -> fc_b (3 elements)

Done! Total parameters: 3603
File saved as '../data/audio/18022026/voice_cnn_weights.h'. Upload this to Vitis HLS.
